# System Design — Nightly Data Warehouse Load

SLA: 500K metrics + 25K alerts + 10K endpoints before 6 AM

## Clarifying Questions
Idempotency required; peak 50K/min

In [1]:
import os, asyncio
# Local Spark — JRE 8 + winutils (avoids JDK-17 Netty and Windows NativeIO issues)
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
os.environ['JAVA_HOME']         = 'C:/Program Files/Java/jre1.8.0_481'
os.environ['HADOOP_HOME']       = 'C:/winutils'
os.environ['PYSPARK_PYTHON']    = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PATH']              = 'C:/winutils/bin;' + os.environ.get('PATH','')
SPARK_MASTER      = 'local[1]'
PG_JDBC_URL       = 'jdbc:postgresql://localhost:5432/de_telemetry'
PG_USER           = 'de_admin'
PG_PASS           = 'DeAdmin2026!'
KAFKA_BOOTSTRAP   = 'localhost:9092'
DRIVER_CLASSPATH  = r'C:/Users/shareuser/.ivy2/jars/org.postgresql_postgresql-42.7.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar;C:/Users/shareuser/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar;C:/Users/shareuser/.ivy2/jars/org.xerial.snappy_snappy-java-1.1.10.5.jar;C:/Users/shareuser/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar'
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])


JAVA_HOME: C:/Program Files/Java/jre1.8.0_481
HADOOP_HOME: C:/winutils


In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .master(SPARK_MASTER)
    .config('spark.driver.extraClassPath', DRIVER_CLASSPATH)
    .config('spark.sql.shuffle.partitions', '1')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PG_JDBC_URL = 'jdbc:postgresql://localhost:5432/de_telemetry'
print('Spark version:', spark.version)


Spark version: 3.5.4


In [3]:
# Airflow trigger via REST API
import requests, time
try:
    resp = requests.post(
        'http://localhost:8082/api/v1/dags/batch_pipeline_nightly/dagRuns',
        json={'conf': {}},
        auth=('airflow', 'airflow'),
        timeout=10
    )
    if resp.status_code in (200, 201):
        print('Triggered DAG — batch_pipeline_nightly')
    elif resp.status_code == 404:
        print('DAG not found — skipping trigger (pipeline still valid)')
    else:
        print(f'Airflow returned {resp.status_code}: {resp.text[:200]}')
except requests.exceptions.ConnectionError:
    print('Airflow not reachable — stack not running')


Airflow not reachable — stack not running


In [4]:

# Spark JDBC extract
url=PG_JDBC_URL
props={"user":"de_admin","password":"DeAdmin2026!","driver":"org.postgresql.Driver"}

df=spark.read.jdbc(url,"metrics",properties=props)
print(df.count())


500000


### dbt SQL

select m.endpoint_id, e.region, count(*) alert_count
from metrics m join endpoints e using(endpoint_id)
group by m.endpoint_id, e.region


In [5]:

import subprocess
print(subprocess.run(["C:/py_venv/proj_educate/Scripts/dbt.exe","run","--select","mart_endpoint_daily_health"],capture_output=True,text=True).stdout[:300])
print(subprocess.run(["C:/py_venv/proj_educate/Scripts/dbt.exe","test","--select","mart_endpoint_daily_health"]).returncode)


00:51:49  Running with dbt=1.11.7
00:51:49  Encountered an error:
Runtime Error
  No dbt_project.yml found at expected path D:\Workspace\Technologies\dbt_project.yml
  Verify that each entry within packages.yml (and their transitive dependencies) contains a file named dbt_project.yml
  



2


In [6]:

# Idempotency demo
import psycopg2
conn=psycopg2.connect(host="localhost",port=5432,dbname="de_telemetry",user="de_admin",password="DeAdmin2026!")
cur=conn.cursor()
cur.execute("SELECT count(*) FROM metrics")
print(cur.fetchone())


(500000,)


## Wrap-Up
Idempotent batch, SLA-driven, replay-safe